# Collaborative Filtering (ALS) + Evaluation
**Dataset**: Amazon Office Products  
**Input**: `rating_rs.parquet` (output từ Part 1)  
**Output**: `als_recommendations.parquet`

## 1. Setup

In [7]:
!pip install pyspark gdown --quiet

In [8]:
RATING_FOLDER_ID = '127zJqoHn2gD3wKlB_oMGC8GnSBL4TzHU'

In [9]:
import gdown

gdown.download_folder(
    id=RATING_FOLDER_ID,
    output='rating_rs.parquet',
    quiet=False
)

Retrieving folder contents


Processing file 1RwqbAWZp4KYsTsFD24k9t1AJGI7ROwJ_ _SUCCESS
Processing file 1o-338s993KpM6D29pjOqTBk6nZ8w8a2F ._SUCCESS.crc
Processing file 1QAijHD2HxPdDIO-UeD1LQ7qAGCMfBsrL .part-00000-f2697475-d0d0-426b-b63f-e9af68adec00-c000.snappy.parquet.crc
Processing file 1pWiR6llaC5Ha3--2w2mE0yFbA_dmlUEX .part-00001-f2697475-d0d0-426b-b63f-e9af68adec00-c000.snappy.parquet.crc
Processing file 1YbQjg-yWnB4eUrSv8rtHlZh2EjDtB6rp .part-00002-f2697475-d0d0-426b-b63f-e9af68adec00-c000.snappy.parquet.crc
Processing file 11zcn1co6GkRMiBzN_raXXBm9zT5FA3yK part-00000-f2697475-d0d0-426b-b63f-e9af68adec00-c000.snappy.parquet
Processing file 1pS9o7mu4MhdeSdvrEBT4muqlkPd-TQw5 part-00001-f2697475-d0d0-426b-b63f-e9af68adec00-c000.snappy.parquet
Processing file 13e3op38ncmeNOFnCRUZeLmL6KZAJ9RYn part-00002-f2697475-d0d0-426b-b63f-e9af68adec00-c000.snappy.parquet


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1RwqbAWZp4KYsTsFD24k9t1AJGI7ROwJ_
To: /content/rating_rs.parquet/_SUCCESS
0.00B [00:00, ?B/s]
Downloading...
From: https://drive.google.com/uc?id=1o-338s993KpM6D29pjOqTBk6nZ8w8a2F
To: /content/rating_rs.parquet/._SUCCESS.crc
100%|██████████| 8.00/8.00 [00:00<00:00, 17.4kB/s]
Downloading...
From: https://drive.google.com/uc?id=1QAijHD2HxPdDIO-UeD1LQ7qAGCMfBsrL
To: /content/rating_rs.parquet/.part-00000-f2697475-d0d0-426b-b63f-e9af68adec00-c000.snappy.parquet.crc
100%|██████████| 243k/243k [00:00<00:00, 16.6MB/s]
Downloading...
From: https://drive.google.com/uc?id=1pWiR6llaC5Ha3--2w2mE0yFbA_dmlUEX
To: /content/rating_rs.parquet/.part-00001-f2697475-d0d0-426b-b63f-e9af68adec00-c000.snappy.parquet.crc
100%|██████████| 244k/244k [00:00<00:00, 8.90MB/s]
Downloading...
From: https://drive.google.com/uc?id=1YbQjg-yWnB4eUrSv8rtHlZh2EjDtB6rp

['rating_rs.parquet/_SUCCESS',
 'rating_rs.parquet/._SUCCESS.crc',
 'rating_rs.parquet/.part-00000-f2697475-d0d0-426b-b63f-e9af68adec00-c000.snappy.parquet.crc',
 'rating_rs.parquet/.part-00001-f2697475-d0d0-426b-b63f-e9af68adec00-c000.snappy.parquet.crc',
 'rating_rs.parquet/.part-00002-f2697475-d0d0-426b-b63f-e9af68adec00-c000.snappy.parquet.crc',
 'rating_rs.parquet/part-00000-f2697475-d0d0-426b-b63f-e9af68adec00-c000.snappy.parquet',
 'rating_rs.parquet/part-00001-f2697475-d0d0-426b-b63f-e9af68adec00-c000.snappy.parquet',
 'rating_rs.parquet/part-00002-f2697475-d0d0-426b-b63f-e9af68adec00-c000.snappy.parquet']

In [10]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName('CollaborativeFiltering')
    .config('spark.driver.memory', '40g')
    .config('spark.sql.shuffle.partitions', '200')
    .getOrCreate()
)
spark

## 2. Load dữ liệu

In [11]:
ratings_raw = spark.read.parquet('rating_rs.parquet')
ratings_raw.printSchema()

root
 |-- product_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- product_parent: string (nullable = true)
 |-- review_rating: double (nullable = true)
 |-- review_date: long (nullable = true)
 |-- verified_purchase: boolean (nullable = true)



In [12]:
ratings_raw.show(5, truncate=False)

+----------+----------------------------+--------------+-------------+-------------+-----------------+
|product_id|user_id                     |product_parent|review_rating|review_date  |verified_purchase|
+----------+----------------------------+--------------+-------------+-------------+-----------------+
|006241710X|AGRH2DK37G3M7S5JLGM2LV7B5ECA|006241710X    |5.0          |1469327339000|true             |
|006241710X|AHBMZZFS2KXFWVQEKJWWQPAY3RZQ|006241710X    |5.0          |1530288228944|true             |
|006241710X|AEOUSA67ZHNDX2SMPVLHLRM5OSTQ|006241710X    |5.0          |1498598110128|true             |
|006241710X|AE4AZYQCS25FCH4KCHVCGH7XET6Q|006241710X    |3.0          |1485227150000|false            |
|006241710X|AGD2SZNHKHI56UBHBYKUXTHPRAJQ|006241710X    |1.0          |1546153378744|true             |
+----------+----------------------------+--------------+-------------+-------------+-----------------+
only showing top 5 rows


## 3. Kiểm tra dữ liệu

In [13]:
print(f'Số rows: {ratings_raw.count():,}')

Số rows: 2,165,547


In [14]:
from pyspark.sql.functions import col, sum as spark_sum, min as spark_min, max as spark_max

ratings_raw.select([
    spark_sum(col(c).isNull().cast('int')).alias(c)
    for c in ['user_id', 'product_id', 'review_rating']
]).show()

+-------+----------+-------------+
|user_id|product_id|review_rating|
+-------+----------+-------------+
|      0|         0|            0|
+-------+----------+-------------+



In [15]:
ratings_raw.select(
    spark_min('review_rating').alias('min_rating'),
    spark_max('review_rating').alias('max_rating')
).show()

+----------+----------+
|min_rating|max_rating|
+----------+----------+
|       1.0|       5.0|
+----------+----------+



In [16]:
ratings_raw.groupBy('review_rating').count().orderBy('review_rating').show()

+-------------+-------+
|review_rating|  count|
+-------------+-------+
|          1.0| 108671|
|          2.0|  70139|
|          3.0| 129510|
|          4.0| 273275|
|          5.0|1583952|
+-------------+-------+



In [17]:
from pyspark.sql.functions import countDistinct

ratings_raw.select(
    countDistinct('user_id').alias('n_users'),
    countDistinct('product_id').alias('n_items')
).show()

+-------+-------+
|n_users|n_items|
+-------+-------+
| 332815| 106583|
+-------+-------+



## 4. Chuan bi du lieu cho ALS


In [18]:
from pyspark.ml.feature import StringIndexer
from pyspark.ml import Pipeline

ratings = ratings_raw.select("user_id", "product_id", "review_rating")

user_indexer = StringIndexer(inputCol="user_id",    outputCol="user_idx",  handleInvalid="skip")
item_indexer = StringIndexer(inputCol="product_id", outputCol="item_idx",  handleInvalid="skip")

pipeline      = Pipeline(stages=[user_indexer, item_indexer])
indexer_model = pipeline.fit(ratings)
ratings_indexed = indexer_model.transform(ratings)

ratings_indexed = ratings_indexed.select(
    col("user_idx").cast("int"),
    col("item_idx").cast("int"),
    col("review_rating")
)

ratings_indexed.show(5)

+--------+--------+-------------+
|user_idx|item_idx|review_rating|
+--------+--------+-------------+
|  198656|   38048|          5.0|
|   53205|   38048|          5.0|
|  288837|   38048|          5.0|
|  148474|   38048|          3.0|
|  257281|   38048|          1.0|
+--------+--------+-------------+
only showing top 5 rows


In [21]:
train, test = ratings_indexed.randomSplit([0.8, 0.2], seed=42)

spark.sparkContext.setCheckpointDir('/tmp/spark-checkpoints')
train = train.checkpoint()
test  = test.checkpoint()

train.cache()
test.cache()

print(f"Train: {train.count():,} rows")
print(f"Test:  {test.count():,} rows")

Train: 1,731,723 rows
Test:  433,824 rows


## 5. Train ALS - Hyperparameter Tuning

In [22]:
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator

evaluator = RegressionEvaluator(
    metricName="rmse",
    labelCol="review_rating",
    predictionCol="prediction"
)

configs = [
    {"rank": 10, "regParam": 0.1,  "maxIter": 10},
    {"rank": 20, "regParam": 0.1,  "maxIter": 10},
    {"rank": 20, "regParam": 0.01, "maxIter": 10},
    {"rank": 50, "regParam": 0.1,  "maxIter": 10},
    {"rank": 50, "regParam": 0.01, "maxIter": 20},
]

results = []

for cfg in configs:
    als = ALS(
        userCol="user_idx",
        itemCol="item_idx",
        ratingCol="review_rating",
        coldStartStrategy="drop",
        nonnegative=True,
        rank=cfg["rank"],
        regParam=cfg["regParam"],
        maxIter=cfg["maxIter"]
    )
    model       = als.fit(train)
    predictions = model.transform(test)
    rmse        = evaluator.evaluate(predictions)
    results.append({**cfg, "rmse": round(rmse, 4)})
    print(f'rank={cfg["rank"]}, regParam={cfg["regParam"]}, maxIter={cfg["maxIter"]} -> RMSE={rmse:.4f}')

rank=10, regParam=0.1, maxIter=10 -> RMSE=1.2455
rank=20, regParam=0.1, maxIter=10 -> RMSE=1.2153
rank=20, regParam=0.01, maxIter=10 -> RMSE=1.8112
rank=50, regParam=0.1, maxIter=10 -> RMSE=1.1921
rank=50, regParam=0.01, maxIter=20 -> RMSE=1.4004


In [23]:
import pandas as pd

results_df = pd.DataFrame(results).sort_values("rmse")
print(results_df.to_string(index=False))

 rank  regParam  maxIter   rmse
   50      0.10       10 1.1921
   20      0.10       10 1.2153
   10      0.10       10 1.2455
   50      0.01       20 1.4004
   20      0.01       10 1.8112


## 6. Train lai voi config tot nhat

In [24]:
best_cfg = results_df.iloc[0]
print(f"Best config: rank={int(best_cfg['rank'])}, regParam={best_cfg['regParam']}, maxIter={int(best_cfg['maxIter'])}, RMSE={best_cfg['rmse']}")

best_als = ALS(
    userCol="user_idx",
    itemCol="item_idx",
    ratingCol="review_rating",
    coldStartStrategy="drop",
    nonnegative=True,
    rank=int(best_cfg["rank"]),
    regParam=float(best_cfg["regParam"]),
    maxIter=int(best_cfg["maxIter"])
)

best_model = best_als.fit(train)

Best config: rank=50, regParam=0.1, maxIter=10, RMSE=1.1921


## 7. Evaluation

In [25]:
# RMSE
predictions = best_model.transform(test)
rmse        = evaluator.evaluate(predictions)
print(f"RMSE: {rmse:.4f}")

RMSE: 1.1921


In [26]:
from pyspark.sql.functions import collect_list, expr, size, array_intersect
from pyspark.mllib.evaluation import RankingMetrics

K = 10

# Ground truth: items user danh gia >= 4 sao trong test set
ground_truth = (
    test.filter(col("review_rating") >= 4.0)
        .groupBy("user_idx")
        .agg(collect_list("item_idx").alias("relevant_items"))
)

# Top-K recommendations
user_recs = best_model.recommendForAllUsers(K)

# Join predictions voi ground truth
eval_df = (
    user_recs
    .join(ground_truth, "user_idx")
    .select(
        expr("transform(recommendations, x -> x.item_idx)").alias("predicted_items"),
        col("relevant_items")
    )
)

# Ranking metrics
rdd             = eval_df.rdd.map(lambda r: (r.predicted_items, r.relevant_items))
ranking_metrics = RankingMetrics(rdd)

precision = ranking_metrics.precisionAt(K)
ndcg      = ranking_metrics.ndcgAt(K)
map_score = ranking_metrics.meanAveragePrecision

# Recall@K
recall_df = eval_df.withColumn(
    "recall",
    size(array_intersect(col("predicted_items"), col("relevant_items"))) / size(col("relevant_items"))
)
recall = recall_df.agg({"recall": "avg"}).collect()[0][0]

print(f"Precision@{K}: {precision:.4f}")
print(f"Recall@{K}:    {recall:.4f}")
print(f"NDCG@{K}:      {ndcg:.4f}")
print(f"MAP:           {map_score:.4f}")

/usr/local/lib/python3.12/dist-packages/pyspark/sql/context.py:157: FutureWarning: Deprecated in 3.0.0. Use SparkSession.builder.getOrCreate() instead.
  warnings.warn(


Precision@10: 0.0006
Recall@10:    0.0044
NDCG@10:      0.0041
MAP:           0.0037


## 8. Export recommendations

In [27]:
from pyspark.sql.functions import explode

# Decode idx -> string
user_labels = indexer_model.stages[0].labels
item_labels = indexer_model.stages[1].labels

idx_to_user = spark.createDataFrame(enumerate(user_labels), ["user_idx", "user_id"])
idx_to_item = spark.createDataFrame(enumerate(item_labels), ["item_idx", "product_id"])

# Flatten va decode
all_recs  = best_model.recommendForAllUsers(10)
recs_flat = (
    all_recs
    .select(col("user_idx"), explode("recommendations").alias("rec"))
    .select("user_idx", col("rec.item_idx").alias("item_idx"), col("rec.rating").alias("als_score"))
    .join(idx_to_user, "user_idx")
    .join(idx_to_item, "item_idx")
    .select("user_id", "product_id", "als_score")
)

recs_flat.show(5)
print(f"Tong recommendations: {recs_flat.count():,}")

+--------------------+----------+---------+
|             user_id|product_id|als_score|
+--------------------+----------+---------+
|AG375WAXLZ7PIOQKI...|B0BS8RYX4T|5.2800283|
|AG375WAXLZ7PIOQKI...|B07B64PCKJ|5.2690787|
|AG375WAXLZ7PIOQKI...|B0B6Q71K64|5.2498045|
|AG375WAXLZ7PIOQKI...|B0BMGDJ4J7|5.2485228|
|AG375WAXLZ7PIOQKI...|B09CTRCJ2X|  5.23839|
+--------------------+----------+---------+
only showing top 5 rows
Tong recommendations: 3,311,900


In [28]:
recs_flat.write.mode("overwrite").parquet("als_recommendations.parquet")

import shutil
from google.colab import files

shutil.make_archive("als_recommendations", "zip", ".", "als_recommendations.parquet")
files.download("als_recommendations.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>